## 1. Instalación y configuración

In [1]:
!pip install -q openai pandas

In [9]:
import os, json, unicodedata, difflib
import pandas as pd
from openai import OpenAI

# --- Clave de API (secretos de Colab o variable de entorno) ---
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise RuntimeError("Falta la clave. Crea el secreto NVIDIA_API_KEY en Colab o la variable de entorno.")

# NVIDIA NIM expone una API compatible con OpenAI
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

# El código busca entre los modelos que NVIDIA lista realmente y PRUEBA cada candidato con una llamada.
# PREFERIDOS son fragmentos de nombre (en minúsculas); se prueban en este orden.
PREFERIDOS = [
    "mistral-large-4", "nemotron-3-ultra", "nemotron-3.5-super", "nemotron-3-5-super",
    "nemotron-3.5-bolt", "nemotron-3-5-bolt", "glm-5", "step-5", "muse-glimmer", "laguna",
]

MODEL = None   # para forzar un modelo, escribe su identificador completo, por ejemplo "empresa/nombre"

HERRAMIENTA_DE_PRUEBA = [{"type": "function", "function": {
    "name": "ping", "description": "Herramienta de prueba.",
    "parameters": {"type": "object", "properties": {}, "required": []}}}]

def elegir_modelo():
    ids = sorted(m.id for m in client.models.list().data)
    print(f"NVIDIA lista {len(ids)} modelos.")
    candidatos = []
    for clave in PREFERIDOS:
        candidatos += [i for i in ids if clave in i.lower() and i not in candidatos]
    print("Candidatos encontrados:", candidatos if candidatos else "ninguno")
    for m in candidatos:
        try:
            client.chat.completions.create(
                model=m, messages=[{"role": "user", "content": "Responde solo: ok"}],
                tools=HERRAMIENTA_DE_PRUEBA, max_tokens=16, temperature=0)
            print(f"  {m}: funciona")
            return m
        except Exception as e:
            print(f"  {m}: no disponible ({str(e)[:120]})")
    raise RuntimeError(
        "Ningún candidato funciona. Modelos que lista NVIDIA: " + ", ".join(ids) +
        "\n\nElige uno con 'Tool Use' en https://build.nvidia.com/models y escribe su identificador en MODEL.")

MODEL = MODEL or elegir_modelo()
print("Modelo en uso:", MODEL)

NVIDIA lista 85 modelos.
Candidatos encontrados: ['mistralai/mistral-large-4-1t-a52b', 'nvidia/nemotron-3-ultra-550b-a55b', 'nvidia/nemotron-3.5-super-vl-120b-a12b', 'nvidia/nemotron-3.5-bolt-e4b', 'z-ai/glm-5.3-flash', 'stepfun-ai/step-5-preview', 'meta/muse-glimmer-30b', 'poolside/laguna-xs-2.1']
  mistralai/mistral-large-4-1t-a52b: funciona
Modelo en uso: mistralai/mistral-large-4-1t-a52b


## 2. Datos

In [3]:
if not os.path.exists("peleadores.csv"):
    from google.colab import files
    files.upload()   # sube peleadores.csv

def normalizar(texto):
    """Minúsculas y sin acentos, para comparar nombres."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()
    return " ".join(t.split())

df = pd.read_csv("peleadores.csv")
df["_clave"] = df["nombre"].map(normalizar)
CATEGORIAS = sorted(df["categoria"].unique())

print(f"{len(df)} peleadores cargados. Categorías: {CATEGORIAS}")
df.drop(columns="_clave").head()

Saving peleadores.csv to peleadores.csv
154 peleadores cargados. Categorías: ['Gallo', 'Ligero', 'Mediano', 'Mosca', 'Pesado', 'Pluma', 'Semipesado', 'Welter']


,nombre,categoria,victorias,derrotas,empates,altura_cm,alcance_cm,peso_lbs
0,Miguel Torres,Gallo,44,9,0,175,193,135
1,Jeff Curran,Gallo,36,18,1,168,175,135
2,Iuri Alcantara,Gallo,35,10,0,175,180,135
3,Urijah Faber,Gallo,35,11,0,168,170,135
4,Jose Aldo,Gallo,31,8,0,170,178,135


## 3. Herramientas

In [4]:
def _buscar(nombre):
    """Devuelve (fila, None) si encuentra un único peleador, o (None, error)."""
    clave = normalizar(nombre)
    exactos = df[df["_clave"] == clave]
    if len(exactos) == 1:
        return exactos.iloc[0], None
    parciales = df[df["_clave"].str.contains(clave, regex=False)] if clave else df.iloc[0:0]
    if len(parciales) == 1:
        return parciales.iloc[0], None
    if len(parciales) > 1:
        return None, {"error": f"'{nombre}' es ambiguo: coincide con varios peleadores.",
                      "coincidencias": parciales["nombre"].tolist()[:10]}
    cercanos = difflib.get_close_matches(clave, df["_clave"].tolist(), n=3, cutoff=0.6)
    return None, {"error": f"No se encontró ningún peleador llamado '{nombre}' en la base de datos.",
                  "sugerencias": df[df["_clave"].isin(cercanos)]["nombre"].tolist()}

def _ficha(f):
    total = int(f.victorias + f.derrotas + f.empates)
    return {
        "nombre": f.nombre, "categoria": f.categoria,
        "victorias": int(f.victorias), "derrotas": int(f.derrotas), "empates": int(f.empates),
        "porcentaje_victorias": round(100 * f.victorias / total, 1) if total else 0.0,
        "altura_cm": int(f.altura_cm), "alcance_cm": int(f.alcance_cm), "peso_lbs": int(f.peso_lbs),
    }

def consultar_peleador(nombre):
    fila, error = _buscar(nombre)
    return error if error else _ficha(fila)

def comparar_peleadores(nombre1, nombre2):
    a, e1 = _buscar(nombre1)
    b, e2 = _buscar(nombre2)
    if e1: return e1
    if e2: return e2
    fa, fb = _ficha(a), _ficha(b)
    ventajas = {}
    # (campo, True si un valor mayor es mejor)
    for campo, mayor_mejor in [("victorias", True), ("derrotas", False), ("porcentaje_victorias", True),
                               ("altura_cm", True), ("alcance_cm", True)]:
        if fa[campo] == fb[campo]:
            ventajas[campo] = "empate"
        else:
            gana_a = (fa[campo] > fb[campo]) == mayor_mejor
            ventajas[campo] = fa["nombre"] if gana_a else fb["nombre"]
    return {"peleador_1": fa, "peleador_2": fb, "ventaja_por_atributo": ventajas}

ORDENES = ["victorias", "derrotas", "altura_cm", "alcance_cm", "peso_lbs"]

def buscar_por_criterio(categoria=None, min_victorias=None, ordenar_por="victorias", n=5):
    r = df
    if categoria:
        cat = {normalizar(c): c for c in CATEGORIAS}.get(normalizar(categoria))
        if cat is None:
            return {"error": f"La categoría '{categoria}' no existe.", "categorias_validas": CATEGORIAS}
        r = r[r["categoria"] == cat]
    if min_victorias is not None:
        r = r[r["victorias"] >= int(min_victorias)]
    if ordenar_por not in ORDENES:
        return {"error": f"No se puede ordenar por '{ordenar_por}'.", "opciones_validas": ORDENES}
    n = max(1, min(int(n), 20))
    total = len(r)
    r = r.sort_values(ordenar_por, ascending=False).head(n)
    return {"total_que_cumplen": total, "mostrando": len(r),
            "resultados": [_ficha(f) for f in r.itertuples()]}

HERRAMIENTAS = {
    "consultar_peleador": consultar_peleador,
    "comparar_peleadores": comparar_peleadores,
    "buscar_por_criterio": buscar_por_criterio,
}

## 4. Descripción de las herramientas para el modelo
El modelo no ve el código: solo ve estas descripciones y los argumentos que recibe cada herramienta.

In [5]:
TOOLS = [
    {"type": "function", "function": {
        "name": "consultar_peleador",
        "description": "Devuelve la ficha de un peleador: categoría, récord (victorias, derrotas, empates), "
                       "porcentaje de victorias, altura, alcance y peso.",
        "parameters": {"type": "object", "properties": {
            "nombre": {"type": "string", "description": "Nombre del peleador, por ejemplo 'Jose Aldo'."}},
            "required": ["nombre"]}}},
    {"type": "function", "function": {
        "name": "comparar_peleadores",
        "description": "Compara dos peleadores en victorias, derrotas, porcentaje de victorias, altura y alcance, "
                       "e indica quién tiene ventaja en cada atributo.",
        "parameters": {"type": "object", "properties": {
            "nombre1": {"type": "string", "description": "Nombre del primer peleador."},
            "nombre2": {"type": "string", "description": "Nombre del segundo peleador."}},
            "required": ["nombre1", "nombre2"]}}},
    {"type": "function", "function": {
        "name": "buscar_por_criterio",
        "description": "Filtra y ordena la tabla de peleadores. Sirve para listas, rankings y preguntas como "
                       "'los 3 con mayor alcance' o 'peleadores de peso ligero con al menos 15 victorias'.",
        "parameters": {"type": "object", "properties": {
            "categoria": {"type": "string", "description": "Categoría de peso: " + ", ".join(CATEGORIAS) + ". Opcional."},
            "min_victorias": {"type": "integer", "description": "Mínimo de victorias (inclusive). Opcional."},
            "ordenar_por": {"type": "string", "enum": ORDENES, "description": "Atributo para ordenar de mayor a menor. Por defecto victorias."},
            "n": {"type": "integer", "description": "Cuántos resultados devolver (máximo 20). Por defecto 5."}},
            "required": []}}},
]

## 5. Agente

In [6]:
SYSTEM_PROMPT = (
    "Eres un asistente experto en datos de peleadores de MMA. Respondes siempre en español, de forma breve y clara. "
    "Todo dato de un peleador debe salir de las herramientas: nunca uses datos de memoria ni inventes cifras. "
    "Si una herramienta devuelve un error o no encuentra datos, díselo al usuario con claridad (puedes ofrecer las "
    "sugerencias que devuelva la herramienta) y no adivines. Si necesitas varios datos, llama a las herramientas "
    "necesarias. Menciona los valores concretos que obtuviste."
)

def _json(obj):
    return json.dumps(obj, ensure_ascii=False)

def ejecutar_herramienta(nombre, argumentos_json):
    """Ejecuta una herramienta pedida por el modelo. Los errores se devuelven como resultado, no se lanzan."""
    try:
        args = json.loads(argumentos_json or "{}")
    except json.JSONDecodeError:
        return {"error": "Los argumentos no son un JSON válido."}, {}
    funcion = HERRAMIENTAS.get(nombre)
    if funcion is None:
        return {"error": f"La herramienta '{nombre}' no existe."}, args
    try:
        return funcion(**args), args
    except TypeError as e:
        return {"error": f"Argumentos incorrectos: {e}"}, args
    except Exception as e:
        return {"error": f"Falló la herramienta: {e}"}, args

MAX_LLAMADAS = 5   # máximo de llamadas a herramientas por pregunta

def agente(pregunta, max_llamadas=MAX_LLAMADAS, verbose=True):
    mensajes = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": pregunta}]
    llamadas = 0
    if verbose:
        print(f"PREGUNTA: {pregunta}\n")

    while True:
        limite = llamadas >= max_llamadas
        respuesta = client.chat.completions.create(
            model=MODEL, messages=mensajes, tools=TOOLS,
            tool_choice="none" if limite else "auto", temperature=0)
        msg = respuesta.choices[0].message

        # Sin herramientas pedidas: es la respuesta final
        if not msg.tool_calls:
            final = msg.content or ""
            if verbose:
                print(f"\nRESPUESTA FINAL: {final}")
            return final

        mensajes.append({
            "role": "assistant", "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls]})

        for tc in msg.tool_calls:
            if llamadas >= max_llamadas:
                resultado, args = {"error": "Límite de llamadas alcanzado. No se ejecutó la herramienta."}, {}
                if verbose:
                    print(f"  [límite de {max_llamadas} llamadas alcanzado] se omite {tc.function.name}")
            else:
                llamadas += 1
                resultado, args = ejecutar_herramienta(tc.function.name, tc.function.arguments)
                if verbose:
                    print(f"  Llamada {llamadas}: {tc.function.name}({args})")
                    print(f"  Resultado: {_json(resultado)[:400]}\n")
            mensajes.append({"role": "tool", "tool_call_id": tc.id, "content": _json(resultado)})

        if llamadas >= max_llamadas:
            mensajes.append({"role": "user", "content":
                "Se alcanzó el límite de llamadas a herramientas. No puedes consultar más. Responde con la "
                "información que ya tienes e indica claramente qué parte de la tarea no pudiste completar."})

## 6. Prueba rápida

In [10]:
agente("¿Cuál es el récord y el alcance de Jose Aldo?")

PREGUNTA: ¿Cuál es el récord y el alcance de Jose Aldo?

  Llamada 1: consultar_peleador({'nombre': 'Jose Aldo'})
  Resultado: {"nombre": "Jose Aldo", "categoria": "Gallo", "victorias": 31, "derrotas": 8, "empates": 0, "porcentaje_victorias": 79.5, "altura_cm": 170, "alcance_cm": 178, "peso_lbs": 135}


RESPUESTA FINAL: Según los datos obtenidos:

**José Aldo**
- **Récord:** 31 victorias, 8 derrotas y 0 empates (79.5% de victorias)
- **Alcance:** 178 cm


'Según los datos obtenidos:\n\n**José Aldo**\n- **Récord:** 31 victorias, 8 derrotas y 0 empates (79.5% de victorias)\n- **Alcance:** 178 cm'

## 7. Casos de prueba

In [14]:
CASOS = [
    ("1. Habitual: consulta de un peleador",
     "¿Cuál es el récord, la altura y el alcance de Urijah Faber?"),
    ("2. Habitual: comparación de dos peleadores",
     "Compara a Jose Aldo e Ilia Topuria. ¿Quién tiene más victorias y mayor alcance?"),
    ("3. Habitual: filtro por criterio",
     "¿Qué peleadores de peso gallo tienen al menos 31 victorias?"),
    ("4. Exige herramientas: encadena dos herramientas",
     "Busca los 2 peleadores de peso gallo con más victorias y compáralos entre sí."),
    ("5. Caso límite: comparar con un peleador que no está en la tabla",
     "Compara a Jose Aldo con Carlos Prates."),
]

resultados = {}
for titulo, pregunta in CASOS:
    print("=" * 80)
    print(titulo)
    print("=" * 80)
    resultados[titulo] = agente(pregunta)
    print()

1. Habitual: consulta de un peleador
PREGUNTA: ¿Cuál es el récord, la altura y el alcance de Urijah Faber?

  Llamada 1: consultar_peleador({'nombre': 'Urijah Faber'})
  Resultado: {"nombre": "Urijah Faber", "categoria": "Gallo", "victorias": 35, "derrotas": 11, "empates": 0, "porcentaje_victorias": 76.1, "altura_cm": 168, "alcance_cm": 170, "peso_lbs": 135}


RESPUESTA FINAL: Aquí están los datos de **Urijah Faber**:

- **Récord:** 35 victorias, 11 derrotas, 0 empates (76.1% de victorias)
- **Altura:** 168 cm
- **Alcance:** 170 cm

Además, pelea en la categoría **Gallo** (135 lbs).

2. Habitual: comparación de dos peleadores
PREGUNTA: Compara a Jose Aldo e Ilia Topuria. ¿Quién tiene más victorias y mayor alcance?

  Llamada 1: comparar_peleadores({'nombre1': 'Jose Aldo', 'nombre2': 'Ilia Topuria'})
  Resultado: {"peleador_1": {"nombre": "Jose Aldo", "categoria": "Gallo", "victorias": 31, "derrotas": 8, "empates": 0, "porcentaje_victorias": 79.5, "altura_cm": 170, "alcance_cm": 178, "p

### Verificación contra el CSV
Para evaluar manualmente, compara cada respuesta con los datos reales del CSV.

In [12]:
cols = ["nombre", "categoria", "victorias", "derrotas", "empates", "altura_cm", "alcance_cm"]
print("Caso 1 y 2:")
display(df[df["nombre"].isin(["Urijah Faber", "Jose Aldo"])][cols])
print("Caso 3 (gallo, >= 31 victorias):")
display(df[(df["categoria"] == "Gallo") & (df["victorias"] >= 31)][cols])
print("Caso 4 (2 gallo con más victorias):")
display(df[df["categoria"] == "Gallo"].sort_values("victorias", ascending=False).head(2)[cols])
print("Caso 5 (debe estar ausente):", df["nombre"].str.contains("Carlos Prates").any())

Caso 1 y 2:


,nombre,categoria,victorias,derrotas,empates,altura_cm,alcance_cm
3,Urijah Faber,Gallo,35,11,0,168,170
4,Jose Aldo,Gallo,31,8,0,170,178


Caso 3 (gallo, >= 31 victorias):


,nombre,categoria,victorias,derrotas,empates,altura_cm,alcance_cm
0,Miguel Torres,Gallo,44,9,0,175,193
1,Jeff Curran,Gallo,36,18,1,168,175
2,Iuri Alcantara,Gallo,35,10,0,175,180
3,Urijah Faber,Gallo,35,11,0,168,170
4,Jose Aldo,Gallo,31,8,0,170,178
5,John Lineker,Gallo,31,9,0,160,170


Caso 4 (2 gallo con más victorias):


,nombre,categoria,victorias,derrotas,empates,altura_cm,alcance_cm
0,Miguel Torres,Gallo,44,9,0,175,193
1,Jeff Curran,Gallo,36,18,1,168,175


Caso 5 (debe estar ausente): False


### Demostración del límite de llamadas
Se baja el límite a 1 para forzar que se alcance. El agente debe responder con lo que obtuvo y avisar que no pudo completar la tarea.

In [13]:
agente("Busca los 2 peleadores de peso gallo con más victorias y compáralos entre sí.", max_llamadas=1)

PREGUNTA: Busca los 2 peleadores de peso gallo con más victorias y compáralos entre sí.

  Llamada 1: buscar_por_criterio({'categoria': 'Gallo', 'ordenar_por': 'victorias', 'n': 2})
  Resultado: {"total_que_cumplen": 10, "mostrando": 2, "resultados": [{"nombre": "Miguel Torres", "categoria": "Gallo", "victorias": 44, "derrotas": 9, "empates": 0, "porcentaje_victorias": 83.0, "altura_cm": 175, "alcance_cm": 193, "peso_lbs": 135}, {"nombre": "Jeff Curran", "categoria": "Gallo", "victorias": 36, "derrotas": 18, "empates": 1, "porcentaje_victorias": 65.5, "altura_cm": 168, "alcance_cm": 175, 


RESPUESTA FINAL: Encontré los **2 peleadores de peso gallo con más victorias**:

1. **Miguel Torres** — 44 victorias, 9 derrotas, 0 empates (83.0% de victorias), altura 175 cm, alcance 193 cm, peso 135 lbs.
2. **Jeff Curran** — 36 victorias, 18 derrotas, 1 empate (65.5% de victorias), altura 168 cm, alcance 175 cm, peso 135 lbs.

**Parte no completada:** no pude ejecutar la herramienta de comparació

'Encontré los **2 peleadores de peso gallo con más victorias**:\n\n1. **Miguel Torres** — 44 victorias, 9 derrotas, 0 empates (83.0% de victorias), altura 175 cm, alcance 193 cm, peso 135 lbs.\n2. **Jeff Curran** — 36 victorias, 18 derrotas, 1 empate (65.5% de victorias), altura 168 cm, alcance 175 cm, peso 135 lbs.\n\n**Parte no completada:** no pude ejecutar la herramienta de comparación entre ambos peleadores porque se alcanzó el límite de llamadas. Por tanto, no dispongo del análisis oficial de quién tiene ventaja en cada atributo (victorias, derrotas, porcentaje, altura y alcance).'

## 8. Preguntar al modelo

In [ ]:
agente(input("Escribe tu pregunta: "))